In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import sys
from pathlib import Path
sys.path.append(str(Path("../src").resolve()))
from quantlab.data.binance import load_parquet
from quantlab.features.technical_features import add_log_returns, add_returns, add_momentum, add_sma, add_rolling_vol
from quantlab.backtest.costs import apply_transaction_costs

In [ ]:
df = load_parquet('../data/raw/BTCUSDT_1d.parquet')
df = add_returns(df)
df = add_momentum(df, 20)
df = add_sma(df, 20)

Let's implement a classic trend following strategy.

A first signal is given by checking the current price vs the price 20 days ago: if it's bigger, we assume it'll keep growing and we buy.

Another strategy is to compare the current price vs a recent mean.

In [ ]:
# Signals
momentum_20d_signal = pd.Series(np.where(df['momentum_20'] > 0, 1, -1), index=df.index)
momentum_moving_average_signal = pd.Series(np.where(df['close'] > df['sma_20'], 1, -1), index=df.index)

In [ ]:
# Strat returns
momentum_20d_return = momentum_20d_signal.shift(1) * df['returns']
momentum_20d_return_net = apply_transaction_costs(momentum_20d_return, momentum_20d_signal, cost_per_trade=0.001)
momentum_moving_average_return = momentum_moving_average_signal.shift(1) * df['returns']
momentum_moving_average_return_net = apply_transaction_costs(momentum_moving_average_return, momentum_moving_average_signal, cost_per_trade=0.001)

In [ ]:
# Equity curves of the two strats vs the buy-and-hold

buy_and_hold_equity = (1 + df['returns']).cumprod()
momentum_20d_equity = (1 + momentum_20d_return).cumprod()
momentum_20d_equity_net = (1 + momentum_20d_return_net).cumprod()
momentum_moving_average_equity = (1 + momentum_moving_average_return).cumprod()
momentum_moving_average_equity_net = (1 + momentum_moving_average_return_net).cumprod()


In [ ]:
plt.figure(figsize=(19, 5))
plt.plot(df.index, buy_and_hold_equity, label = 'B&H')
plt.plot(df.index, momentum_20d_equity, label = 'Momentum 20d')
plt.plot(df.index, momentum_20d_equity_net, label = 'Momentum 20d /w costs')
plt.plot(df.index, momentum_moving_average_equity, label = 'Momentum Moving Average')
plt.plot(df.index, momentum_moving_average_equity_net, label = 'Momentum Moving Average /w costs')
plt.title('Equity Curves with different strategies')

plt.grid(True)
plt.legend()

In [ ]:
momentum_20d_return.describe()

In [ ]:
df['returns'].describe()

In [ ]:
plt.figure(figsize=(19,7))

plt.plot(df['close'], label = df['close'].name)

for i in range(len(df) - 1):
    if momentum_20d_signal.iloc[i] == 1:
        plt.axvspan(df.index[i], df.index[i+1], color = 'green', alpha = 0.2)
    else:
        plt.axvspan(df.index[i], df.index[i+1], color = 'red', alpha = 0.2)

plt.legend()
plt.grid(True)

In [ ]:
momentum_20d_signal.value_counts()

In [ ]:
momentum_moving_average_signal.value_counts()

There are a lot of swings, which might mean that the signal is noisy: let's try to put a threshold.

In [ ]:
threshold = 0.02

momentum_20d_signal_threshold = pd.Series([1 if x > threshold else -1 if x < - threshold else 0 for x in df['momentum_20']], index = df.index)
momentum_moving_average_signal_threshold = pd.Series([1 if x > 1 + threshold else -1 if x < 1 - threshold else 0 for x in df['close']/df['sma_20'] - 1], index = df.index)

momentum_20d_return_threshold = momentum_20d_signal_threshold.shift(1) * df['returns']
momentum_20d_return_threshold_net = apply_transaction_costs(momentum_20d_return_threshold, momentum_20d_signal_threshold, cost_per_trade=0.001)

momentum_moving_average_return_threshold = momentum_moving_average_signal_threshold.shift(1) * df['returns']
momentum_moving_average_return_threshold_net = apply_transaction_costs(momentum_moving_average_return_threshold, momentum_moving_average_signal_threshold, cost_per_trade=0.001)

momentum_20d_equity_threshold = (1 + momentum_20d_return_threshold).cumprod()
momentum_20d_equity_threshold_net = (1 + momentum_20d_return_threshold_net).cumprod()

momentum_moving_average_equity_threshold = (1 + momentum_moving_average_return_threshold).cumprod()
momentum_moving_average_equity_threshold_net = (1 + momentum_moving_average_return_threshold_net).cumprod()

In [ ]:
fig, axes = plt.subplots(5, figsize=(19, 20))

axes[0].plot(df.index, momentum_20d_equity, label= 'Momentum 20d')
axes[0].plot(df.index, momentum_20d_equity_net, label= 'Momentum 20d /w costs')
axes[0].plot(df.index, momentum_20d_equity_threshold, label= 'Momentum 20d /w threshold')
axes[0].plot(df.index, momentum_20d_equity_threshold_net, label = 'Momentum 20d /w threshold and costs')
axes[0].set_title('Impact of threshold and costs on Equity Curves for Momentum 20d')
axes[0].grid(True)
axes[0].legend()

axes[1].plot(df.index, momentum_moving_average_equity, label = 'Momentum Moving Average')
axes[1].plot(df.index, momentum_moving_average_equity_net, label='Momentum Moving Average /w costs')
axes[1].plot(df.index, momentum_moving_average_equity_threshold, label = 'Momentum Moving Average /w threshold')
axes[1].plot(df.index, momentum_moving_average_equity_threshold_net, label='Momentum Moving Average /w threshold and costs')
axes[1].set_title('Impact of threshold and costs on Equity Curves for Momentum Moving Average')
axes[1].grid(True)
axes[1].legend()

axes[2].plot(df.index, buy_and_hold_equity, label= 'B&H')
axes[2].plot(df.index, momentum_20d_equity_threshold_net, label = 'Momentum 20d')
axes[2].plot(df.index, momentum_moving_average_equity_threshold_net, label='Momentum Moving Average')
axes[2].set_title('Equity Curves for momentum 20d and momentum moving avg. (with threshold and costs) vs B&H')
axes[2].grid(True)
axes[2].legend()

axes[3].plot(df.index, df['close'], label='close prices')
for i in range(len(df) - 1):
    if momentum_20d_signal_threshold.iloc[i] == 1:
        axes[3].axvspan(df.index[i], df.index[i+1], color = 'green', alpha = 0.2)
    if momentum_20d_signal_threshold.iloc[i] == -1:
        axes[3].axvspan(df.index[i], df.index[i+1], color = 'red', alpha = 0.2)
axes[3].set_title('momentum_20d_signal_threshold buy (green), sell (red), no action (no color) pattern')
axes[3].grid(True)
axes[3].legend()

axes[4].plot(df.index, df['close'], label='close prices')
for i in range(len(df) - 1):
    if momentum_moving_average_signal_threshold.iloc[i] == 1:
        axes[4].axvspan(df.index[i], df.index[i+1], color = 'green', alpha = 0.2)
    if momentum_moving_average_signal_threshold.iloc[i] == -1:
        axes[4].axvspan(df.index[i], df.index[i+1], color = 'red', alpha = 0.2)

axes[4].set_title('momentum_moving_average_signal_threshold buy (green), sell (red), no action (no color) pattern')
axes[4].grid(True)
axes[4].legend()

plt.tight_layout()

In [ ]:
momentum_20d_signal_threshold.value_counts()

In [ ]:
momentum_moving_average_signal_threshold.value_counts()